In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from utils import (
    load_config_json,
    scale_config_to_resolution,
    load_and_preprocess_image,
    get_or_compute_vesselness,
    get_or_detect_aorta_circles,
    get_or_segment_aorta,
    detect_and_evaluate_ostia,
    visualize_aorta_with_ostia,
    segment_arteries_from_ostia,
    visualize_arteries_comparison,
)

# Análise de Casos Ruins

In [2]:
# Configurações
CONFIG_PATH = "../config/pipeline_config.json"
DATA_PATH = "/media/matheus/HD/DatasetsCCTA/ImageCAS/1-1000" # "/data04/home/mpmaia/ImageCAS/database/1-1000"
OUTPUT_PATH = "../output"
USE_CACHE = True
HTML_OUTPUT_DIR = Path(f"{OUTPUT_PATH}/cases_analysis_3d")
HTML_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 42

# Carregar config padrão do arquivo com uma base config mínima
base_config = {}
CONFIG = load_config_json(CONFIG_PATH, base_config)

# Carregar dados de bad cases
bad_cases_high = pd.read_csv(f"{OUTPUT_PATH}/segmentation/8.final_results/bad_cases_exports/bad_cases_test_high_res.csv")
bad_cases_mid = pd.read_csv(f"{OUTPUT_PATH}/segmentation/8.final_results/bad_cases_exports/bad_cases_test_mid_res.csv")

# Adicionar coluna de resolução
bad_cases_high['resolution'] = 'high'
bad_cases_mid['resolution'] = 'mid'

# Combinar datasets
all_bad_cases = pd.concat([bad_cases_high, bad_cases_mid], ignore_index=True)

# Mostrar tipos de erros e suas frequências por resolução
print("Distribuição de tipos de erro por resolução:")
for resolution in ['high', 'mid']:
    print(f"\n{resolution.upper()} Resolution:")
    res_data = all_bad_cases[all_bad_cases['resolution'] == resolution]
    print(res_data['bad_case_status'].value_counts())

print()

# Configuração de amostras por tipo de erro e resolução
np.random.seed(RANDOM_SEED)
n_samples_per_error_per_resolution = 3  # 3 amostras de cada tipo de erro para cada resolução

# Agrupar por tipo de erro E resolução, e amostrar de cada combinação
selected_cases_list = []
for error_type, grp in all_bad_cases.groupby('bad_case_status'):
    high_grp = grp[grp['resolution'] == 'high']
    mid_grp = grp[grp['resolution'] == 'mid']

    # ids disponíveis por resolução
    high_ids = set(high_grp['image_id'].astype(int).tolist())
    mid_ids = set(mid_grp['image_id'].astype(int).tolist())

    # identificar ids compartilhados (mesma imagem com mesmo status em ambas resoluções)
    shared_ids = sorted(list(high_ids & mid_ids))

    # escolher até n amostras compartilhadas
    k_shared = min(len(shared_ids), n_samples_per_error_per_resolution)
    shared_sample_ids = []
    if k_shared > 0:
        shared_sample_ids = list(np.random.choice(shared_ids, size=k_shared, replace=False))

    # adicionar linhas compartilhadas (uma por resolução) para cada id compartilhado selecionado
    for img in shared_sample_ids:
        row_high = high_grp[high_grp['image_id'].astype(int) == img].iloc[0:1]
        row_mid = mid_grp[mid_grp['image_id'].astype(int) == img].iloc[0:1]
        selected_cases_list.append(row_high)
        selected_cases_list.append(row_mid)

    # para cada resolução, preencher o restante das amostras com exemplos não compartilhados
    for resolution, res_grp in [('high', high_grp), ('mid', mid_grp)]:
        already_ids = set(shared_sample_ids)
        remaining_needed = n_samples_per_error_per_resolution - len(shared_sample_ids)
        if remaining_needed <= 0:
            print(f"Selecionados {len(shared_sample_ids)} compartilhados para erro '{error_type}' em ambas resoluções; nenhum adicional necessário para {resolution}.")
            continue
        available = res_grp[~res_grp['image_id'].astype(int).isin(already_ids)]
        n_to_sample = min(remaining_needed, len(available))
        if n_to_sample > 0:
            sample = available.sample(n=n_to_sample, random_state=RANDOM_SEED)
            selected_cases_list.append(sample)
            print(f"Selecionados {n_to_sample} não-compartilhados para erro '{error_type}' em resolução {resolution.upper()}.")
        else:
            print(f"Nenhum caso adicional disponível para erro '{error_type}' em resolução {resolution.upper()}.")

    if len(shared_sample_ids) == 0:
        print(f"Nenhum id compartilhado para erro '{error_type}' entre HIGH e MID.")
    else:
        print(f"Selecionados {len(shared_sample_ids)} ids compartilhados para erro '{error_type}': {shared_sample_ids}")

# concatenar todos os resultados selecionados
if selected_cases_list:
    selected_cases = pd.concat(selected_cases_list, ignore_index=True)
else:
    selected_cases = pd.DataFrame(columns=all_bad_cases.columns)

print(f"\nTotal de {len(selected_cases)} casos selecionados para análise")
print("\nCasos Selecionados:")
print(selected_cases[['image_id', 'bad_case_status', 'resolution']].sort_values(['bad_case_status', 'resolution']))

Distribuição de tipos de erro por resolução:

HIGH Resolution:
bad_case_status
none_correct       98
one_correct        65
ostia_not_found    37
low_dice           27
Name: count, dtype: int64

MID Resolution:
bad_case_status
none_correct       87
low_dice           27
one_correct        17
ostia_not_found     6
Name: count, dtype: int64

Selecionados 3 compartilhados para erro 'low_dice' em ambas resoluções; nenhum adicional necessário para high.
Selecionados 3 compartilhados para erro 'low_dice' em ambas resoluções; nenhum adicional necessário para mid.
Selecionados 3 ids compartilhados para erro 'low_dice': [np.int64(171), np.int64(595), np.int64(632)]
Selecionados 3 compartilhados para erro 'none_correct' em ambas resoluções; nenhum adicional necessário para high.
Selecionados 3 compartilhados para erro 'none_correct' em ambas resoluções; nenhum adicional necessário para mid.
Selecionados 3 ids compartilhados para erro 'none_correct': [np.int64(916), np.int64(372), np.int64(683)]
S

In [ ]:
def run_pipeline_for_case(img_id, config, data_path=None, save_base_path=None):
    """Executa o pipeline completo para uma imagem e retorna os resultados"""
    if data_path is None:
        data_path = str(DATA_PATH)
    print(f"\n{'='*60}")
    print(f"Processando IMG {img_id}")
    print(f"{'='*60}")

    try:
        resolution = 'high' if config['DOWNSCALE_FACTORS'] == [1, 1, 1] else 'mid'
        cache_root = save_base_path or f"{OUTPUT_PATH}/cases_analysis_cache"
        vesselness_cache_dir = f"{cache_root}/vesselness_{resolution}"
        pipeline_cache_dir = f"{cache_root}/pipeline_{resolution}"
        artery_cache_dir = f"{cache_root}/artery_{resolution}"

        run_config = dict(config)
        run_config['LOAD_CACHE'] = USE_CACHE
        run_config['SAVE_CACHE'] = USE_CACHE

        # 1. Carregar e pré-processar
        print("1. Carregando e pré-processando a imagem...")
        image_data = load_and_preprocess_image(img_id, data_path, run_config)
        lcc_image = image_data['lcc_image']
        label = image_data['label']
        downscale_factors = image_data['downscale_factors']
        scaled_spacing = image_data['scaled_spacing']

        del image_data # Liberar memória

        # 2. Detectar circulos aorta
        print("2. Detectando óstios (círculos na aorta)...")
        detected_circles = get_or_detect_aorta_circles(
            img_id, lcc_image, downscale_factors, scaled_spacing,
            run_config['CIRCLE_DETECTION'], base_save_path=pipeline_cache_dir,
            load_cache=USE_CACHE, save_cache=USE_CACHE
        )

        # 3. Segmentar aorta
        print("3. Segmentando aorta...")
        aorta_mask = get_or_segment_aorta(
            img_id, lcc_image, detected_circles,
            run_config['LEVEL_SET'], pipeline_cache_dir,
            load_cache=USE_CACHE, save_cache=USE_CACHE
        )

        # 4. Calcular vesselness para ostios
        print("4. Calculando vesselness para ostios...")
        vesselness_ostios = get_or_compute_vesselness(
            img_id, lcc_image, cache_dir=vesselness_cache_dir,
            vesselness_config=run_config['VESSELNESS_AORTA'],
            load_cache=USE_CACHE, save_cache=USE_CACHE
        )

        # 5. Detectar e avaliar óstios
        print("5. Detectando e avaliando óstios...")
        try:
            ostia_results = detect_and_evaluate_ostia(
                aorta_mask, vesselness_ostios, label, scaled_spacing, run_config
            )
            ostia_left = ostia_results['ostia_left']
            ostia_right = ostia_results['ostia_right']
            label_artery = ostia_results['label_artery']
        except ValueError as ostia_exc:
            print(f"Erro na detecção/avaliação dos óstios: {ostia_exc}")
            ostia_left = None
            ostia_right = None
            label_artery = None
            ostia_results = None

        del vesselness_ostios  # Liberar memória
        del detected_circles # Liberar memória

        # 6. Segmentar artérias
        print("6. Segmentando artérias a partir dos óstios...")
        artery_results = segment_arteries_from_ostia(
            img_id, lcc_image, label_artery, ostia_left, ostia_right,
            run_config, artery_cache_dir
        )

        print(f"Pipeline completado com sucesso para a imagem {img_id}!\n")

        return {
            'image_id': img_id,
            'label_artery': label_artery,
            'aorta_mask': aorta_mask,
            'ostia_left': ostia_left,
            'ostia_right': ostia_right,
            'artery_results': artery_results,
            'ostia_results': ostia_results,
            'scaled_spacing': scaled_spacing
        }

    except Exception as e:
        print(f"Erro ao processar IMG {img_id}: {e}")
        import traceback
        traceback.print_exc()
        return {'success': False, 'error': str(e)}

print("Funcao de pipeline definida")

Funcao de pipeline definida


: 

## Visualizar Óstios + Aorta + Label

In [ ]:
pipeline_results = {}

for idx, row in selected_cases.iterrows():
    img_id = int(row['image_id'])
    resolution = row['resolution']
    bad_case_status = row['bad_case_status']

    if resolution == 'high':
        CONFIG["DOWNSCALE_FACTORS"] = [1, 1, 1]
    else:
        CONFIG["DOWNSCALE_FACTORS"] = [2, 2, 1]

    # Escalar config para a resolução base da configuração
    scaled_config = scale_config_to_resolution(CONFIG)

    # Executar pipeline
    result = run_pipeline_for_case(img_id, scaled_config)
    pipeline_results[img_id] = result

    print(f"\nIMG {img_id} ({resolution} res) - Status: {bad_case_status}")

    # Sanitizar o nome do tipo de erro para usar no nome do arquivo
    error_type_clean = bad_case_status.lower().replace(' ', '_').replace('/', '_')

    # Informações dos óstios
    ostia_left = result['ostia_left']
    ostia_right = result['ostia_right']

    print(f"Óstios detectados - Esquerdo: {len(ostia_left)}, Direito: {len(ostia_right)}")

    # Salvar aorta com óstios em HTML
    aorta_mask = result['aorta_mask']
    label = result['label']
    spacing = result['scaled_spacing']

    # Gerar visualização 3D da aorta com os óstios detectados
    try:
        ostia_filename = f"img_{img_id}_{resolution}_{error_type_clean}_aorta_ostios.html"
        ostia_path = HTML_OUTPUT_DIR / ostia_filename
        visualize_aorta_with_ostia(
            aorta_mask,
            ostia_left,
            ostia_right,
            spacing=spacing,
            label_mask=label,
            use_physical_coords=True,
            save_html_path=ostia_path,
            display_plot=False,
        )
        print(f"   ✓ HTML salvo: {ostia_filename}")
    except Exception as e:
        print(f"   ✗ Erro ao gerar HTML de óstios: {e}")

    # Gerar visualização comparativa das artérias segmentadas vs label
    try:
        artery_filename = f"img_{img_id}_{resolution}_{error_type_clean}_arteries_comparison.html"
        artery_path = HTML_OUTPUT_DIR / artery_filename
        artery_mask = result['artery_results']['artery_mask']
        visualize_arteries_comparison(
            label_mask=label,
            predicted_mask=artery_mask,
            spacing=spacing,
            save_html_path=artery_path,
            display_plot=False,
        )
        print(f"   ✓ HTML salvo: {artery_filename}")
    except Exception as e:
        print(f"   ✗ Erro ao gerar HTML de segmentação: {e}")

print("\nVisualizações de Detecção de Óstios + Aorta concluídas")


Processando IMG 171
1. Carregando e pré-processando a imagem...
2. Detectando óstios (círculos na aorta)...
Parada na fatia 147: Δr=2.50 ou dist=52.88
3. Segmentando aorta...
Array salvo em: ../output/cases_analysis_cache/pipeline_high/segmented_aorta/171_mask_aorta.npy
4. Calculando vesselness para ostios...
